# Week 7 Exercise - Base vs Fine-tuned: Does Fine-tuning Actually Help?

This notebook loads both the **base Llama 3.1-8B** and **ed-donner's fine-tuned variant**, then runs them head-to-head on price prediction to measure the actual improvement from fine-tuning.

**Runs on Google Colab with T4 GPU.**

In [ ]:
!pip install -q --upgrade torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 --index-url https://download.pytorch.org/whl/cu124
!pip install -q --upgrade bitsandbytes>=0.46.1 transformers>=4.48.3 accelerate>=1.3.0 datasets>=3.2.0 peft>=0.14.0 gradio>=4.0

In [ ]:
import re
import random
from tqdm import tqdm
import numpy as np
from google.colab import userdata
from huggingface_hub import login
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import PeftModel
from datasets import load_dataset

login(userdata.get('HF_TOKEN'), add_to_git_credential=True)

In [ ]:
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
FINETUNED_MODEL = "ed-donner/pricer-2024-09-13_13.04.39"
REVISION = "e8d637df551603dc86cd7a1598a8f44af4d7ae36"

dataset = load_dataset("ed-donner/home-data")
test = dataset["test"]
print(f"Test set: {len(test)} items")

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config, device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Wrap the base model with PEFT adapter for the fine-tuned version
ft_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL, revision=REVISION)
ft_model.eval()
print("Both models ready")

## Prediction Functions

Two approaches:
- **Base model**: disable the adapter so we get raw Llama predictions
- **Fine-tuned model**: enable the adapter to get the trained predictions

In [ ]:
def extract_price(text):
    """Pull a dollar amount out of generated text."""
    text = text.replace(",", "")
    if "Price is $" in text:
        text = text.split("Price is $")[1]
    match = re.search(r"\d+\.?\d*", text)
    return float(match.group()) if match else 0.0


def predict(prompt, use_adapter=True):
    """Generate a price prediction. Toggle adapter on/off for base vs fine-tuned."""
    set_seed(42)
    if use_adapter:
        ft_model.enable_adapter_layers()
    else:
        ft_model.disable_adapter_layers()

    inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda")
    attention_mask = torch.ones(inputs.shape, device="cuda")
    with torch.no_grad():
        outputs = ft_model.generate(
            inputs, attention_mask=attention_mask, max_new_tokens=5, num_return_sequences=1,
        )
    return extract_price(tokenizer.decode(outputs[0]))

In [ ]:
# Quick test on first item
item = test[0]
print(item["text"][:300])
print(f"\nActual:     ${item['price']:.2f}")
print(f"Base model: ${predict(item['text'], use_adapter=False):.2f}")
print(f"Fine-tuned: ${predict(item['text'], use_adapter=True):.2f}")

## Head-to-Head Evaluation

In [ ]:
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"


def run_evaluation(use_adapter, label, data, size=50):
    """Evaluate a model variant and return errors."""
    errors = []
    print(f"\n--- {label} ---")
    for i in tqdm(range(min(size, len(data)))):
        truth = data[i]["price"]
        guess = predict(data[i]["text"], use_adapter=use_adapter)
        error = abs(guess - truth)
        errors.append(error)

        if error < 40 or error / truth < 0.2:
            color = GREEN
        elif error < 80 or error / truth < 0.4:
            color = YELLOW
        else:
            color = RED
        print(f"{color}${error:.0f}{RESET} ", end="")

    mae = np.mean(errors)
    median = np.median(errors)
    print(f"\nMAE: ${mae:.2f} | Median error: ${median:.2f}")
    return errors

In [ ]:
base_errors = run_evaluation(use_adapter=False, label="Base Llama 3.1-8B", data=test, size=50)
ft_errors = run_evaluation(use_adapter=True, label="Fine-tuned Llama 3.1-8B", data=test, size=50)

improvement = np.mean(base_errors) - np.mean(ft_errors)
print(f"\nFine-tuning improvement: ${improvement:.2f} lower MAE")

## Gradio UI - Side by Side Comparison

In [ ]:
import gradio as gr


def compare_predictions(description):
    """Run both models on the same input and show the difference."""
    prompt = f"What does this cost to the nearest dollar?\n\n{description}\n\nPrice is $"
    base_price = predict(prompt, use_adapter=False)
    ft_price = predict(prompt, use_adapter=True)
    return f"${base_price:,.2f}", f"${ft_price:,.2f}"


def random_test_item():
    """Load a random test item for quick testing."""
    item = random.choice(list(test))
    # Strip the prompt wrapper, keep just the product description
    text = item["text"]
    if "\n\n" in text:
        parts = text.split("\n\n")
        description = "\n\n".join(parts[1:-1])  # middle part is the description
    else:
        description = text
    return description, f"${item['price']:.2f}"


with gr.Blocks(title="Base vs Fine-tuned Price Predictor") as demo:
    gr.Markdown("# Base vs Fine-tuned Llama 3.1-8B Price Predictor")
    gr.Markdown("Compare how fine-tuning changes the model's price estimates.")

    with gr.Row():
        description = gr.Textbox(lines=6, label="Product Description")

    with gr.Row():
        predict_btn = gr.Button("Compare Models", variant="primary")
        random_btn = gr.Button("Random Test Item")

    actual_price = gr.Textbox(label="Actual Price (for test items)", interactive=False)

    with gr.Row():
        base_output = gr.Textbox(label="Base Llama 3.1-8B")
        ft_output = gr.Textbox(label="Fine-tuned Llama 3.1-8B")

    predict_btn.click(fn=compare_predictions, inputs=description, outputs=[base_output, ft_output])
    random_btn.click(fn=random_test_item, outputs=[description, actual_price])

demo.launch()

## Conclusion

By toggling the PEFT adapter on and off, we can directly measure the impact of fine-tuning on the same base model. The fine-tuned model should show a meaningfully lower MAE, confirming that task-specific training improves price prediction even on a quantized 8B model.